# 09 - Confidence Threshold Analizi

Farklı güvenilirlik eşiklerinde Precision-Recall trade-off.

## Analiz
- P-R eğrisi
- Optimal threshold belirleme
- Araç tespiti ve plaka OCR için ayrı analiz

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

In [ ]:
# Farklı confidence threshold'larla deney çalıştır
from src.pipeline.pipeline_factory import create_pipeline

THRESHOLDS = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
VIDEO = 'data/videos/test/sample.mp4'

threshold_results = []

for thresh in THRESHOLDS:
    print(f'Threshold: {thresh}')
    try:
        pipeline = create_pipeline(
            'configs/config.yaml',
            overrides={
                'general.video_source': VIDEO,
                'vehicle_detection.confidence_threshold': thresh,
                'general.save_video': False,
                'general.output_dir': f'results/thresh_{thresh}',
            }
        )
        stats = pipeline.run()
        stats['threshold'] = thresh
        threshold_results.append(stats)
    except Exception as e:
        print(f'  Hata: {e}')

In [ ]:
if threshold_results:
    df = pd.DataFrame(threshold_results)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # İhlal sayısı vs threshold
    axes[0].plot(df['threshold'], df['total_violations'], 'b-o')
    axes[0].set_xlabel('Confidence Threshold')
    axes[0].set_ylabel('Tespit Edilen İhlal Sayısı')
    axes[0].set_title('Threshold vs İhlal Sayısı')
    axes[0].grid(True)
    
    # FPS vs threshold
    axes[1].plot(df['threshold'], df['average_fps'], 'r-o')
    axes[1].set_xlabel('Confidence Threshold')
    axes[1].set_ylabel('FPS')
    axes[1].set_title('Threshold vs FPS')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('results/threshold_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    df.to_csv('results/threshold_analysis.csv', index=False)
    print(df[['threshold', 'total_violations', 'average_fps']].to_string(index=False))